# Mutli Hop RAG

In [ ]:
!pip install langchain-chroma langchain-huggingface langsmith langchain-openai langchain-core

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import shutil
import zipfile
import torch
from google.colab import drive, userdata
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# LLM 및 체인 구성을 위한 추가 라이브러리
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableLambda

# Pydantic Output Parser 구성 라이브러리
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import List, Optional

# API Key 설정
try:
    os.environ["LANGCHAIN_API_KEY"] = userdata.get('langgrpah')
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
    os.environ["LANGCHAIN_PROJECT"] = "multi-hop-rag"

    # OpenRouter 키 (변수명 확인 필요)
    openrouter_key = userdata.get('openrouter')
except Exception as e:
    print(f"Key 설정 오류: {e}")

print("환경 설정 완료")

환경 설정 완료


In [ ]:
# 데이터 경로 설정 (설정경로 꼭 바꿀 것!)
LOCAL_EXTRACT_PATH = "/content/drive/MyDrive/chroma_db_bge_m3"

# 데이터 셋업 함수
def setup_data():
    if os.path.exists(LOCAL_EXTRACT_PATH) and os.listdir(LOCAL_EXTRACT_PATH):
        print(f"로컬 데이터 경로 확인됨: {LOCAL_EXTRACT_PATH}")
        return True
    else:
        print(f"오류: 경로에 데이터가 없습니다 -> {LOCAL_EXTRACT_PATH}")
        return False

# Pydantic 스키마 정의 (Reasoning 포함)
class Evidence(BaseModel):
    author: str = Field(description="The author of the source document")
    category: str = Field(description="The category of the document")
    fact: str = Field(description="Exactly one single key sentence extracted from content")
    published_at: str = Field(description="The publication date")
    source: str = Field(description="The source name")
    title: str = Field(description="The title of the document")
    url: str = Field(description="The URL of the source")

class MultiHopResponse(BaseModel):
    reasoning: str = Field(description="Step-by-step logical deduction process.")
    Answer: str = Field(description="The final answer (Short sentence, Yes, No, or Insufficient information)")
    evidence_list: List[Evidence] = Field(description="List of supporting evidence items")

if setup_data():
    print("데이터 및 스키마 준비 완료")

로컬 데이터 경로 확인됨: /content/drive/MyDrive/chroma_db_bge_m3
데이터 및 스키마 준비 완료


In [ ]:
SYSTEM_PROMPT = """
You are an expert AI assistant capable of performing multi-hop reasoning based on provided documents.
Your goal is to answer the user's query accurately and provide structured evidence derived strictly from the Context.

### 1. PROCESS (CHAIN OF THOUGHT)
**CRITICAL:** Before formulating the final "Answer", you must first write down your logical deduction in the "reasoning" field.
- Explain how you connected the facts from different [Document ID] blocks.
- Verify if the found information fully answers the query.

### 2. GUIDELINES
- **Evidence Extraction:** For every fact, extract **exactly one single key sentence** verbatim.
- **Handle 'No':** If documents prove the query false, answer "No" with contradictory evidence.
- **Handle 'Insufficient':** If information is missing, answer "Insufficient information" and leave evidence list empty.

### 3. OUTPUT FORMAT (STRICT JSON)
You must output ONLY a valid JSON object.

{{
  "reasoning": "Write your step-by-step logic here. e.g., 'Doc 0 mentions A, and Doc 1 confirms A causes B, therefore the answer is B.'",
  "Answer": "Final Answer ('Yes', 'No', 'Short Answer', or 'Insufficient information')",
  "evidence_list": [
    {{
      "author": "Value from Metadata",
      "category": "Value from Metadata",
      "fact": "Exactly one single sentence from Content.",
      "published_at": "Value from Metadata",
      "source": "Value from Metadata",
      "title": "Value from Metadata",
      "url": "Value from Metadata"
    }}
  ]
}}

### 4. CONSTRAINTS
- The "fact" field must contain **only one single sentence**.
- Do not fabricate metadata.
"""

In [ ]:
# Retrieval
def get_global_retriever(persist_dir):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading Embedding Model on {device}...")

    embedding_model = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": True}
    )

    real_db_path = persist_dir
    for root, dirs, files in os.walk(persist_dir):
        if "chroma.sqlite3" in files:
            real_db_path = root
            break

    vectorstore = Chroma(
        persist_directory=real_db_path,
        embedding_function=embedding_model,
        collection_name="multihop_rag"
    )

    return vectorstore.as_retriever(search_kwargs={"k": 3})

# Chain 생성
def create_chain():
    # Retriever 로드
    retriever = get_global_retriever(LOCAL_EXTRACT_PATH)

    # LLM 설정
    llm = ChatOpenAI(
        model="openai/gpt-oss-120b:free",
        openai_api_key=userdata.get('openrouter'),
        openai_api_base="https://openrouter.ai/api/v1",
        temperature=0, # 사실 기반 답변
        default_headers={
            "HTTP-Referer": "https://colab.research.google.com",
            "X-Title": "MULTI HOP RAG Project"
        }
    )

    # Doc Formatting
    # 검색된 문서들을 하나의 문자열로 합치는 함수
    def format_docs_with_metadata(docs):
      formatted_string = ""
      for i, doc in enumerate(docs):
          # 메타데이터 추출 (없으면 'N/A' 처리)
          source = doc.metadata.get('source', 'N/A')
          title = doc.metadata.get('original_title', 'N/A')
          author = doc.metadata.get('author', 'N/A')
          date = doc.metadata.get('published_at', 'N/A')
          url = doc.metadata.get('url', 'N/A')
          category = doc.metadata.get('category', 'N/A')

          # LLM이 읽기 쉬운 블록 형태로 구성
          formatted_string += f"""
          [Document ID: {i}]
          Metadata:
          - Source: {source}
          - Title: {title}
          - Author: {author}
          - Published: {date}
          - URL: {url}
          - Category: {category}
          Content:
          {doc.page_content}
          --------------------------------------------------
          """
      return formatted_string

    # Reasoning 필터
    def filter_reasoning(data):
        if isinstance(data, dict) and "reasoning" in data:
            del data["reasoning"]
        return data

    # Chain 조립
    parser = JsonOutputParser(pydantic_object=MultiHopResponse)

    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", "Context:\n{context}\n\nQuestion:\n{question}")
    ])

    # RAG 체인 연결
    rag_chain = (
        {"context": retriever | format_docs_with_metadata, "question": RunnablePassthrough()}
        | prompt
        | llm
        | parser
        | RunnableLambda(filter_reasoning) # reasoing output에 안나오게 필터
    )

    return rag_chain

# 실행
print("모델 및 체인 로딩 시작... (잠시만 기다려주세요)")
global_rag_chain = create_chain()
print("로딩 완료!")

모델 및 체인 로딩 시작... (잠시만 기다려주세요)
Loading Embedding Model on cuda...
로딩 완료!


In [ ]:
from sklearn.model_selection import train_test_split
from datasets import load_dataset
from tqdm import tqdm
import ast
import re
import string
from collections import Counter

# ==============================================================================
# 1. 데이터 샘플링 (Evaluation Set Generation)
# ==============================================================================
print("1. QA 데이터셋 로드 및 샘플링 중...")

# 데이터셋 로드
try:
    ds = load_dataset("yixuantt/MultiHopRAG", "MultiHopRAG", split="train")
    qa_df = ds.to_pandas()
except Exception as e:
    print(f"데이터셋 로드 실패: {e}")
    # 만약 이미 로드된 qa_df가 있다면 사용
    # qa_df = ...

# 비율 맞춰서 50개 추출 (Stratified Sampling)
sampled_df, _ = train_test_split(
    qa_df,
    train_size=50,
    stratify=qa_df['question_type'],
    random_state=42
)
print(f"   -> 평가용 데이터 {len(sampled_df)}개 추출 완료")

1. QA 데이터셋 로드 및 샘플링 중...


README.md: 0.00B [00:00, ?B/s]

MultiHopRAG.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2556 [00:00<?, ? examples/s]

   -> 평가용 데이터 50개 추출 완료


In [ ]:
# ==============================================================================
# 2. RAG 모델 추론 (Inference)
# ==============================================================================
print("\n2. RAG 모델 추론 시작 (global_rag_chain 사용)...")

rag_answers = []
rag_evidence_lists = []

# tqdm으로 진행률 표시
for index, row in tqdm(sampled_df.iterrows(), total=len(sampled_df), desc="Inference"):
    query = row['query']

    try:
        # [핵심] 사용자님의 체인 호출
        response = global_rag_chain.invoke(query)

        # 결과 파싱 (Pydantic 모델 출력 가정)
        # response는 딕셔너리 형태 {'answer': ..., 'evidence_list': ...}
        ans = response.get('Answer', "No Answer")
        ev_list = response.get('evidence_list', [])

        rag_answers.append(ans)
        rag_evidence_lists.append(ev_list)

    except Exception as e:
        print(f"   [Error] Index {index}: {e}")
        rag_answers.append("Error")
        rag_evidence_lists.append([])

# 결과 DataFrame 생성
eval_df = sampled_df[['query', 'evidence_list', 'answer']].copy()
eval_df['RAG_evidence_list'] = rag_evidence_lists
eval_df['RAG_answer'] = rag_answers

# 컬럼 순서 정리
basic_eval_df = eval_df[["query", "evidence_list", "RAG_evidence_list", "answer", "RAG_answer"]]

#저장
basic_eval_df.to_csv("/content/drive/MyDrive/eval_df.csv", index=False, encoding="utf-8-sig")
print("   -> 추론 완료. 데이터프레임 생성됨.")



2. RAG 모델 추론 시작 (global_rag_chain 사용)...


Inference: 100%|██████████| 50/50 [04:44<00:00,  5.68s/it]

   -> 추론 완료. 데이터프레임 생성됨.


In [ ]:
import pandas as pd
import ast
import re

# 1. 데이터 로드
df = pd.read_csv("/content/drive/MyDrive/eval_df.csv")

# ------------------------------------------------------------------
# [Helper] 리스트 파싱 함수 (데이터 포맷 오류 자동 보정)
# ------------------------------------------------------------------
def parse_list_robust(x):
    """
    CSV 저장 과정에서 리스트 내 쉼표가 누락된 경우를 처리하여 파싱합니다.
    예: "{...}\n {...}" -> "{...}, {...}"
    """
    if not isinstance(x, str): return []

    # 1차 시도: 일반적인 파싱
    try:
        return ast.literal_eval(x)
    except:
        pass

    # 2차 시도: 쉼표 누락 보정 (Regex)
    # 닫는 중괄호 } 와 여는 중괄호 { 사이에 쉼표가 없으면 추가
    fixed_str = re.sub(r'\}\s*\{', '}, {', x)

    try:
        return ast.literal_eval(fixed_str)
    except:
        return [] # 여전히 실패하면 빈 리스트 반환

# 리스트 컬럼 파싱 적용
df['evidence_list'] = df['evidence_list'].apply(parse_list_robust)
df['RAG_evidence_list'] = df['RAG_evidence_list'].apply(parse_list_robust)

# ------------------------------------------------------------------
# [Core] 평가지표 계산 함수 (Hit, MRR, MAP)
# ------------------------------------------------------------------
def calculate_retrieval_metrics(row, k=5):
    """
    한 행(Query)에 대해 Hit@K, MRR@K, MAP@K를 계산합니다.
    """
    # 1. Gold Evidence (정답) 추출 - URL 기준 Set (중복 제거, O(1) 탐색)
    gold_urls = set([item.get('url') for item in row['evidence_list'] if item.get('url')])

    # 2. Retrieved Evidence (예측) 추출 - URL 기준 List (순서 유지)
    # 중복된 문서가 검색될 경우, 상위 순위 1개만 남기고 제거 (Deduplication)
    raw_retrieved_urls = [item.get('url') for item in row['RAG_evidence_list'] if item.get('url')]
    retrieved_urls = []
    seen = set()
    for url in raw_retrieved_urls:
        if url not in seen:
            retrieved_urls.append(url)
            seen.add(url)

    # Top-K 자르기
    retrieved_urls = retrieved_urls[:k]

    # -------------------------------------------------------
    # 지표 1: Hit@K (하나라도 찾았는가?)
    # -------------------------------------------------------
    # 교집합이 있으면 1, 없으면 0
    hit = 1 if not gold_urls.isdisjoint(retrieved_urls) else 0

    # -------------------------------------------------------
    # 지표 2: MRR@K (첫 정답은 몇 번째인가?)
    # -------------------------------------------------------
    mrr = 0
    for i, url in enumerate(retrieved_urls):
        if url in gold_urls:
            mrr = 1 / (i + 1) # Rank는 1부터 시작
            break # 첫 번째 정답만 보고 종료

    # -------------------------------------------------------
    # 지표 3: MAP@K (여러 정답을 얼마나 잘 찾았는가?)
    # -------------------------------------------------------
    num_gold = len(gold_urls) # |G|: 전체 정답 개수

    if num_gold == 0:
        ap = 0 # 정답이 아예 없는 데이터는 0 처리
    else:
        hits = 0
        sum_precisions = 0

        for i, url in enumerate(retrieved_urls):
            if url in gold_urls:
                hits += 1 # 맞춘 개수 증가
                precision_at_i = hits / (i + 1) # 현재 위치까지의 정밀도
                sum_precisions += precision_at_i

        # Average Precision = (Precision 합) / (전체 정답 개수 |G|)
        ap = sum_precisions / num_gold

    return pd.Series([hit, mrr, ap], index=[f'Hit@{k}', f'MRR@{k}', f'MAP@{k}'])

# ------------------------------------------------------------------
# [Execution] 전체 데이터프레임에 적용
# ------------------------------------------------------------------
K_VALUE = 5 # Top-5 기준

# 각 행별 점수 계산
metrics_df = df.apply(lambda row: calculate_retrieval_metrics(row, k=K_VALUE), axis=1)

# 원본 데이터와 합치기
final_df = pd.concat([df, metrics_df], axis=1)

# ------------------------------------------------------------------
# [Result] 결과 확인
# ------------------------------------------------------------------
print(f"======== Evaluation Results (K={K_VALUE}) ========")
print(final_df[[f'Hit@{K_VALUE}', f'MRR@{K_VALUE}', f'MAP@{K_VALUE}']].mean())

# 상세 결과 저장
final_df.to_csv("/content/drive/MyDrive/basic_rag_evaluation_with_metrics.csv", index=False)

======== Evaluation Results (K=5) ========
Hit@5    0.420000
MRR@5    0.400000
MAP@5    0.273333
dtype: float64


In [1]:
# @title 평가지표
import pandas as pd
import ast
import re
import string
from collections import Counter
import os
from google.colab import drive, userdata


# 1. 구글 드라이브 마운트 (데이터가 드라이브에 있으므로 필수)
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 1. 파일 로드 (경로는 사용자 환경에 맞게 유지)
df = pd.read_csv("/content/drive/MyDrive/eval_df.csv", encoding="utf-8-sig")

# ------------------------------------------------------------------
# [Helper] 파싱 함수
# ------------------------------------------------------------------
def parse_list_robust(x):
    if not isinstance(x, str): return []
    try:
        return ast.literal_eval(x)
    except:
        pass
    fixed_str = re.sub(r'\}\s*\{', '}, {', x)
    try:
        return ast.literal_eval(fixed_str)
    except:
        return []

df['evidence_list'] = df['evidence_list'].apply(parse_list_robust)
df['RAG_evidence_list'] = df['RAG_evidence_list'].apply(parse_list_robust)

# ------------------------------------------------------------------
# [Metric 1] Retrieval Metrics (Hit, MRR, MAP)
# ------------------------------------------------------------------
def calculate_retrieval_metrics(row, k=5):
    gold_urls = set([item.get('url') for item in row['evidence_list'] if item.get('url')])
    raw_retrieved_urls = [item.get('url') for item in row['RAG_evidence_list'] if item.get('url')]

    # 중복 제거 (순서 유지)
    retrieved_urls = []
    seen = set()
    for url in raw_retrieved_urls:
        if url not in seen:
            retrieved_urls.append(url)
            seen.add(url)
    retrieved_urls = retrieved_urls[:k]

    # Hit@K
    hit = 1 if not gold_urls.isdisjoint(retrieved_urls) else 0

    # MRR@K
    mrr = 0
    for i, url in enumerate(retrieved_urls):
        if url in gold_urls:
            mrr = 1 / (i + 1)
            break

    # MAP@K
    num_gold = len(gold_urls)
    if num_gold == 0:
        ap = 0
    else:
        hits = 0
        sum_precisions = 0
        for i, url in enumerate(retrieved_urls):
            if url in gold_urls:
                hits += 1
                sum_precisions += hits / (i + 1)
        ap = sum_precisions / num_gold

    return pd.Series([hit, mrr, ap], index=[f'Hit@{k}', f'MRR@{k}', f'MAP@{k}'])

# [Metric 2] Answer Match Accuracy (Exact Match & F1)
def normalize_answer(s):
    """
    평가를 위해 답변 텍스트를 정규화합니다.
    (소문자 변환, 문장부호 제거)
    """

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return str(text).lower()

    if not s or pd.isna(s): return ""

    return white_space_fix(remove_punc(lower(s)))

def calculate_qa_metrics(row):
    """
    정답(Answer)과 모델 예측(RAG_Answer)을 비교합니다.
    """
    gold_text = normalize_answer(row['answer'])
    pred_text = normalize_answer(row['RAG_answer'])

    # Exact Match (EM): 완전히 일치하는가? (0 or 1)
    em = 1 if gold_text == pred_text else 0


    return pd.Series([em], index=['Exact Match Accuracy'])

# ------------------------------------------------------------------
# [Execution] 전체 적용 및 저장
# ------------------------------------------------------------------
K_VALUE = 5

# 1. Retrieval Score 계산
retrieval_scores = df.apply(lambda row: calculate_retrieval_metrics(row, k=K_VALUE), axis=1)

# 2. QA Score 계산 (추가된 부분)
qa_scores = df.apply(calculate_qa_metrics, axis=1)

# 3. 전체 데이터프레임 병합
final_df = pd.concat([df, retrieval_scores, qa_scores], axis=1)

# 결과 출력 (평균 점수 확인)
print(f"======== Evaluation Results (K={K_VALUE}) ========")
# 백분율(%)로 변환하여 출력
summary = final_df[[f'Hit@{K_VALUE}', f'MRR@{K_VALUE}', f'MAP@{K_VALUE}', 'Exact Match Accuracy']].mean() * 100
print(summary)

# CSV 저장
save_path = "/content/drive/MyDrive/rag_evaluation_with_all_metrics_mj_basic.csv"
final_df.to_csv(save_path, index=False, encoding="utf-8-sig")
print(f"\n[Done] 결과 파일이 저장되었습니다: {save_path}")

Mounted at /content/drive
======== Evaluation Results (K=5) ========
Hit@5                   42.000000
MRR@5                   40.000000
MAP@5                   27.333333
Exact Match Accuracy    48.000000
dtype: float64

[Done] 결과 파일이 저장되었습니다: /content/drive/MyDrive/rag_evaluation_with_all_metrics_mj_basic.csv


## Inference Query

In [ ]:
# @title inference_query
USER_QUERY = "Which individual is implicated in both inflating the value of a Manhattan apartment to a figure not yet achieved in New York City's real estate history, according to 'Fortune', and is also accused of adjusting this apartment's valuation to compensate for a loss in another asset's worth, as reported by 'The Age'?"

In [ ]:
# 실행
try:
    print("\n" + "="*50)
    print(f"질문: {USER_QUERY}")
    print("="*50)
    print("답변 생성 중... (LangSmith 추적 중)")

    # 이미 로드된 global_rag_chain 사용
    response = global_rag_chain.invoke(USER_QUERY)

    print("\n[답변]")
    print(response)

    # print("\n완료. LangSmith 대시보드에서 Trace를 확인하세요.")

except Exception as e:
    print(f"\n실행 중 오류 발생: {e}")


질문: Which individual is implicated in both inflating the value of a Manhattan apartment to a figure not yet achieved in New York City's real estate history, according to 'Fortune', and is also accused of adjusting this apartment's valuation to compensate for a loss in another asset's worth, as reported by 'The Age'?
답변 생성 중... (LangSmith 추적 중)

[답변]
{'Answer': 'Donald Trump', 'evidence_list': [{'author': 'N/A', 'category': 'N/A', 'fact': 'No apartment in New York City has ever sold for close to that amount, James said.', 'published_at': '2023-09-26 21:11:15.000000000Z', 'source': 'Fortune', 'title': "Donald Trump defrauded banks with 'fantasy' to build his real estate empire, judge rules in a major repudiation against the former president", 'url': 'https://fortune.com/2023/09/26/donald-trump-fraud-banks-insurers-real-estate-judge-new-york/'}, {'author': 'N/A', 'category': 'N/A', 'fact': 'The prosecution argues that was to mask a drop in the value of one of his other properties.', 'pub

In [ ]:
USER_QUERY = "Who is the individual associated with the cryptocurrency industry facing a criminal trial on fraud and conspiracy charges, as reported by both The Verge and TechCrunch, and is accused by prosecutors of committing fraud for personal gain?"

In [ ]:
# 실행
try:
    print("\n" + "="*50)
    print(f"질문: {USER_QUERY}")
    print("="*50)
    print("답변 생성 중... (LangSmith 추적 중)")

    # 이미 로드된 global_rag_chain 사용
    response = global_rag_chain.invoke(USER_QUERY)

    print("\n[답변]")
    print(response)

    # print("\n완료. LangSmith 대시보드에서 Trace를 확인하세요.")

except Exception as e:
    print(f"\n실행 중 오류 발생: {e}")


질문: Who is the individual associated with the cryptocurrency industry facing a criminal trial on fraud and conspiracy charges, as reported by both The Verge and TechCrunch, and is accused by prosecutors of committing fraud for personal gain?
답변 생성 중... (LangSmith 추적 중)

[답변]
{'Answer': 'Sam Bankman-Fried', 'evidence_list': [{'author': 'N/A', 'category': 'N/A', 'fact': 'The FTX trial is bigger than Sam Bankman-Fried.', 'published_at': '2023-09-28 12:00:00.000000000Z', 'source': 'The Verge', 'title': 'The FTX trial is bigger than Sam Bankman-Fried', 'url': 'https://www.theverge.com/2023/9/28/23893269/ftx-sam-bankman-fried-trial-evidence-crypto'}, {'author': 'N/A', 'category': 'N/A', 'fact': 'Sam Bankman-Fried, founder of former Coinbase rival FTX, was found guilty of seven criminal fraud counts tied to the collapse of his exchange and the theft of customer funds.', 'published_at': '2023-11-30 21:08:00.000000000Z', 'source': 'Cnbc | World Business News Leader', 'title': 'Coinbase rallies

## Comparison Query

In [ ]:
# @title comparison query (Yes)
USER_QUERY = "Do the TechCrunch article on software companies and the Hacker News article on The Epoch Times both report an increase in revenue related to payment and subscription models, respectively?"

In [ ]:
# 실행
try:
    print("\n" + "="*50)
    print(f"질문: {USER_QUERY}")
    print("="*50)
    print("답변 생성 중... (LangSmith 추적 중)")

    # 이미 로드된 global_rag_chain 사용
    response = global_rag_chain.invoke(USER_QUERY)

    print("\n[답변]")
    print(response)

    # print("\n완료. LangSmith 대시보드에서 Trace를 확인하세요.")

except Exception as e:
    print(f"\n실행 중 오류 발생: {e}")


질문: Do the TechCrunch article on software companies and the Hacker News article on The Epoch Times both report an increase in revenue related to payment and subscription models, respectively?
답변 생성 중... (LangSmith 추적 중)

[답변]
{'Answer': 'No', 'evidence_list': [{'author': 'N/A', 'category': 'N/A', 'fact': 'TechCrunch+ subscribers get access to in-depth commentary, analysis and surveys — which you know if you’re already a subscriber.', 'published_at': '2023-12-09 21:16:17.000000000Z', 'source': 'TechCrunch', 'title': 'Google fakes an AI demo, Grand Theft Auto VI goes viral and Spotify cuts jobs', 'url': 'https://techcrunch.com/2023/12/09/google-fakes-an-ai-demo-grand-theft-auto-vi-goes-viral-and-spotify-cuts-jobs/'}, {'author': 'N/A', 'category': 'N/A', 'fact': 'The Epoch Times reported $8.4 million in revenue from contributions and grants in 2020 and 2021.', 'published_at': '2023-10-16 03:41:24.000000000Z', 'source': 'Hacker News', 'title': 'How the conspiracy-fueled Epoch Times went m

In [ ]:
# @title comparison query (No)
USER_QUERY = "Does the TechCrunch article suggest that the success in ""North America's EV market"" is due to the size and price of electric vehicles, while The Verge article focuses on Donald Trump's criticism of electric vehicles regarding their cost, range, and impact on American jobs?"

In [ ]:
# 실행
try:
    print("\n" + "="*50)
    print(f"질문: {USER_QUERY}")
    print("="*50)
    print("답변 생성 중... (LangSmith 추적 중)")

    # 이미 로드된 global_rag_chain 사용
    response = global_rag_chain.invoke(USER_QUERY)

    print("\n[답변]")
    print(response)

    # print("\n완료. LangSmith 대시보드에서 Trace를 확인하세요.")

except Exception as e:
    print(f"\n실행 중 오류 발생: {e}")


질문: Does the TechCrunch article suggest that the success in North America's EV market is due to the size and price of electric vehicles, while The Verge article focuses on Donald Trump's criticism of electric vehicles regarding their cost, range, and impact on American jobs?
답변 생성 중... (LangSmith 추적 중)

[답변]
{'Answer': 'Insufficient information', 'evidence_list': []}


In [ ]:
USER_QUERY = "Does 'The Age' article suggest that Australia's Davis Cup team is aiming for an improvement in their performance compared to the previous year, while the 'Sporting News' article indicates that the South Africa national rugby team has already achieved an improvement to reach the Rugby World Cup semi-finals?"

In [ ]:
# 실행
try:
    print("\n" + "="*50)
    print(f"질문: {USER_QUERY}")
    print("="*50)
    print("답변 생성 중... (LangSmith 추적 중)")

    # 이미 로드된 global_rag_chain 사용
    response = global_rag_chain.invoke(USER_QUERY)

    print("\n[답변]")
    print(response)

    # print("\n완료. LangSmith 대시보드에서 Trace를 확인하세요.")

except Exception as e:
    print(f"\n실행 중 오류 발생: {e}")


질문: Does 'The Age' article suggest that Australia's Davis Cup team is aiming for an improvement in their performance compared to the previous year, while the 'Sporting News' article indicates that the South Africa national rugby team has already achieved an improvement to reach the Rugby World Cup semi-finals?
답변 생성 중... (LangSmith 추적 중)

[답변]
{'Answer': 'Insufficient information', 'evidence_list': []}


## Temporal Query

In [ ]:
# @title temporal_query (No)
USER_QUERY = "Did The Independent - Sports report on the All Blacks' home victories against Ireland, South Africa, and Argentina last summer on October 14, 2023, and did The Roar | Sports Writers Blog report on Argentina's victories over the All Blacks in Christchurch last year and their first victory in 2020 in Sydney on October 18, 2023, making the reporting on the All Blacks' defeats by Argentina consistent?"

In [ ]:
# 실행
try:
    print("\n" + "="*50)
    print(f"질문: {USER_QUERY}")
    print("="*50)
    print("답변 생성 중... (LangSmith 추적 중)")

    # 이미 로드된 global_rag_chain 사용
    response = global_rag_chain.invoke(USER_QUERY)

    print("\n[답변]")
    print(response)

    # print("\n완료. LangSmith 대시보드에서 Trace를 확인하세요.")

except Exception as e:
    print(f"\n실행 중 오류 발생: {e}")


질문: Did The Independent - Sports report on the All Blacks' home victories against Ireland, South Africa, and Argentina last summer on October 14, 2023, and did The Roar | Sports Writers Blog report on Argentina's victories over the All Blacks in Christchurch last year and their first victory in 2020 in Sydney on October 18, 2023, making the reporting on the All Blacks' defeats by Argentina consistent?
답변 생성 중... (LangSmith 추적 중)

[답변]
{'Answer': 'Insufficient information', 'evidence_list': []}


## Null Query

In [ ]:
# @title null query
USER_QUERY = "Considering the information from a BBC article detailing Sridevi's achievements in the Indian film industry and a Times of India report on her posthumous honors, which single character from a film portrayed by Sridevi has been recognized for its cultural impact and has also been commemorated with a special award after her passing?"

In [ ]:
# 실행
try:
    print("\n" + "="*50)
    print(f"질문: {USER_QUERY}")
    print("="*50)
    print("답변 생성 중... (LangSmith 추적 중)")

    # 이미 로드된 global_rag_chain 사용
    response = global_rag_chain.invoke(USER_QUERY)

    print("\n[답변]")
    print(response)

    # print("\n완료. LangSmith 대시보드에서 Trace를 확인하세요.")

except Exception as e:
    print(f"\n실행 중 오류 발생: {e}")


질문: Considering the information from a BBC article detailing Sridevi's achievements in the Indian film industry and a Times of India report on her posthumous honors, which single character from a film portrayed by Sridevi has been recognized for its cultural impact and has also been commemorated with a special award after her passing?
답변 생성 중... (LangSmith 추적 중)

[답변]
{'Answer': 'Insufficient information', 'evidence_list': []}
